# Ascend C算子调用介绍
本节讲解Ascend C算子调用相关内容，将为你详细拆解单算子API与pybind11两种调用方式的实现流程与实操方法。我们将按照以下结构，带大家系统掌握Ascend C算子的调用技巧：
- 环境准备
- Ascend C算子调用方式有哪些
- 如何开发单算子API调用算子
- 如何开发pybind11调用算子
- 课后实践
---

## **1.环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及代码能编译运行。

In [ ]:
!mkdir -p Sources/03.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

本章节为算子调用，需要提前部署好已开发的算子，请在开始学习前执行以下操作完成算子部署。

- 部署算子工程

上一节课中，我们已经完成了 AddCustomTemplate 算子工程的开发讲解。本节课将继续以该算子为例，详细介绍算子的调用流程与使用方法。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/03.03/custom_op

# 复制上一节课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op Sources/03.03/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/03.03/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/03.03/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

---
## **2. Ascend C算子调用方式有哪些**
Ascend C算子常见的调用方式有kernel直调、单算子调用、第三方框架中调用。工程化的算子通常以单算子调用和第三方框架中调用为主。具体调用方式和调用条件如下：

<table style="float: left; border-collapse: collapse; margin: 0; background: #f9f9f9; font-size: 14px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;">
    <tr>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">调用方式</th>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">使用条件</th>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子API</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子模型执行</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">IR构图</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pytorch框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">TensorFlow框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
        <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pybind调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
</table>  

<div style="clear: both;"></div>
<p style="margin-top: 10px; line-height: 1.6;">
   这节课我们主要讲解工程化算子的单算子API和Pybind调用。
</p>

---  

## **3. 如何使用单算子API调用算子**
单算子API调用算子基于C语言的API执行算子，直接调用单算子API接口，无需提供单算子描述文件进行离线模型的转换，是算子交付阶段最重要的一种调用方式，也是Ascend C算子开发人员必须掌握的算子调用手段。  

Ascend C算子开发并编译部署完成后，会在```$ASCEND_TOOLKIT_HOME/opp/vendors```目录下的自定义算子vendor_name/op_api目录下会自动生成单算子API，以默认安装场景为例，单算子调用的头文件.h和动态库libcust_opapi.so所在的目录结构为：

<pre>
├── opp                  // 算子库目录
│   ├── vendors          // 自定义算子所在目录
│   │   ├── config.ini
│   │   └── customize // 自定义算子安装包时配置的vendor_name，默认值为customize
│   │       ├── op_api
│   │       │   ├── include
│   │       │   │   └── aclnn_xx.h
│   │       │   └── lib
│   │       │       └── libcust_opapi.so
</pre>

执行以下命令，查看我们已经部署好的算子目录

In [ ]:
!cd ${HOME}/vendors;find . -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

算子调用头文件通常为aclnn_+下划线命名法算子名+.h，以[工程化算子开发介绍](./03.02_operator_engineering_intro.ipynb)章节内的AddCustomTemplate为例，我们可以在op_api/include目录下找到aclnn_add_custom_template.h文件，该文件为AddCustomTemplate算子的单算子调用头文件。  

aclnn_xx.h中的算子API形式一般定义为“两段式接口”，形如：

```
aclnnStatus aclnnXxxGetWorkspaceSize(const aclTensor *输入1, const aclTensor *输入2, ......, aclTensor *输出1, aclTensor *输出2, ......, uint64_t workspaceSize, aclOpExecutor **executor); 

aclnnStatus aclnnXxx(void* workspace, int64 workspaceSize, aclOpExecutor* executor, aclrtStream stream);
```

以AddCustomTemplate算子为例，我们可以在aclnn_add_custom_template.h文件中找到AddCustomTemplate算子的二段式调用接口：

```
aclnnStatus aclnnAddCustomTemplateGetWorkspaceSize(const aclTensor *x, const aclTensor *y, const aclTensor *z, uint64_t *workspaceSize, aclOpExecutor **executor);

aclnnStatus aclnnAddCustomTemplate(void *workspace, uint64_t workspaceSize, aclOpExecutor *executor, aclrtStream stream);
```

单算子API可以直接在应用程序中调用，大致过程为：

1. 使用第一段接口aclnnXxxGetWorkspaceSize计算本次API调用计算过程中需要多少的workspace内存

2. 获取到本次API计算需要的workspace大小后，按照workspaceSize大小申请Device侧内存

3. 调用第二段接口aclnnXxx，调用对应的单算子二进制执行计算  

完整调用流程：  
<img src="./images/single_operator_api_invocation_flow.png" alt="single_operator_api_invocation_flow" style="zoom:80%;" />

下面我们以AddCustomTemplate算子为例，介绍单算子API的调用流程。  

### 3.1 头文件引用
安装部署完成后，会在算子包安装目录下的op_api目录生成单算子调用的头文件aclnn_xx.h和动态库libcust_opapi.so，编写单算子的调用代码时，要包含自动生成的单算子API执行接口头文件，即aclnn_add_custom_template.h：

In [ ]:
%%writefile Sources/03.03/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_add_custom_template.h"

引用头文件后，我们还可以定义一些工具函数，简化代码的编写，例如定义CHECK_ACL宏，用于检查ACL API调用是否成功：


In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)


### 3.2 ACL初始化和运行资源申请
初始化ACL环境，申请运行资源，这部分代码为固定写法。

In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));


### 3.3 申请输入输出数据内存
在调用单算子API前，需要先申请内存，用于存储算子的输入输出数据。首先定义输入输出数据的shape，根据shape获取应该申请的内存大小。接着调用aclrtMalloc申请内存，申请成功后创建aclTensor承载该内存，创建Tensor时指定了Tensor的shape、dtype、format。

In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp    
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(aclFloat16);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);

### 3.4 构造测试数据
上一步骤我们已经申请了输入输出数据的内存，现在需要构造测试数据，填充到申请的内存中。
这里我们直接使用1和2填充输入vector，实际应用中通常是从文件或者其他算子输出中读取数据。
构造数据后，我们需要将数据填充到申请的内存中，这里使用aclrtMemcpy将数据从host内存复制到device内存。

In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp      
    std::vector<aclFloat16> input0HostData(elementCount, aclFloatToFloat16(1.0));
    std::vector<aclFloat16> input1HostData(elementCount, aclFloatToFloat16(2.0));
    std::vector<aclFloat16> output0HostData(elementCount, aclFloatToFloat16(0.0));
    std::vector<aclFloat16> goldenData(elementCount, aclFloatToFloat16(3.0));

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));

### 3.5 计算workspace并申请内存
在执行算子前，需要先使用第一段接口aclnnXxxGetWorkspaceSize计算本次API调用计算过程中需要多少的workspace内存，然后根据计算workspace大小申请内存。

In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnAddCustomTemplateGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

### 3.6 执行算子并处理结果
申请workspace内存后，我们可以调用aclnnAddCustomTemplate执行算子计算。并使用aclrtSynchronizeStream等待流所有操作执行完成，确保算子计算完成。计算结果会存储在输出Tensor中，我们需要从device内存复制到host内存，再进行打印或者保存为文件的处理。


In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp
    CHECK_ACL(aclnnAddCustomTemplate(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", aclFloat16ToFloat(output0HostData[i])); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");

### 3.7 释放内存、运行资源和去初始化
在完成算子计算后，需要释放申请的内存、运行资源和去初始化ACL环境。需要释放的内存和上文申请的内存一一对应。释放内存完成后再释放运行资源，最后去初始化ACL环境。

In [ ]:
%%writefile -a Sources/03.03/aclnn_test.cpp
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

这样我们就完成了AddCustomTemplate算子的调用，输入是shape为[8, 2048]的float16类型数据，输出是shape为[8, 2048]的float16类型数据。尝试运行一下，查看输出结果是否符合预期。

In [ ]:
# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/03.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/03.03/execute_add_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/03.03/execute_add_op

调用程序正常执行后会有如下打印：
```

result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0
test pass

```

---

## **4. 如何使用pybind调用算子**
Pybind是一个用于将C++代码与Python解释器集成的库，实现原理是通过将C++代码编译成动态链接库（DLL）或共享对象（SO）文件，使用Pybind提供的API将算子aclnn接口与Python解释器进行绑定。在Python解释器中使用绑定的C++函数、类和变量，从而实现Python与C++代码的交互。  

本质上，pybind调用算子仍是通过调用aclnn接口实现算子计算，只是将aclnn接口封装成Python函数，方便在Python中调用。

使用pybind调用算子的基本流程如下：  
<img src="./images/pybind_process_flow.png"  alt="pybind_process_flow" />

### 4.1 pybind环境准备
在进行pybind调用算子前，需要准备好pybind调用算子的环境，执行以下命令安装pybind调用算子的依赖：

In [ ]:
!pip install torch==2.9.0;pip install torch-npu==2.9.0;pip install pybind11;pip install setuptools; pip install wheel

### 4.2 pybind封装代码编写
仍旧以调用AddCustomTemplate算子为例，我们在部署算子后，单算子API即可调用。我们需要编写pybind封装代码，将aclnnAddCustomTemplate接口封装成Python函数，供Python调用。  
#### 4.2.1 头文件引用
首先引用必要的头文件，其中[pytorch_npu_helper.hpp](./src/pybind_op_03/pytorch_npu_helper.hpp)内定义了宏EXEC_NPU_CMD用于执行aclnn二段式接口。

In [ ]:
# 清理可能存在的pybind工程
!rm -rf Sources/03.03/pybind_op
# 复制已准备好的工程
!cp -r ./src/pybind_op_03 Sources/03.03/pybind_op

In [ ]:
%%writefile Sources/03.03/pybind_op/custom_op.cpp

#include <torch/extension.h>
#include <torch/csrc/autograd/custom_function.h>
#include "pytorch_npu_helper.hpp"
using torch::autograd::Function;
using torch::autograd::AutogradContext;
using tensor_list = std::vector<at::Tensor>;
using namespace at;

#### 4.2.2 算子调用函数编写
由于工具类中已经定义了EXEC_NPU_CMD宏，我们可以直接使用该宏调用aclnnAddCustomTemplate接口。我们可以定义一个npu_add_custom_template函数，接收输入张量x和y，然后EXEC_NPU_CMD宏完成算子调用，然后返回输出张量z。

In [ ]:
%%writefile -a Sources/03.03/pybind_op/custom_op.cpp
at::Tensor npu_add_custom_template(const at::Tensor &x, const at::Tensor &y) {
    at::Tensor z = at::empty_like(x);
    EXEC_NPU_CMD(aclnnAddCustomTemplate, x, y, z);
    return z;
}

#### 4.2.3 绑定算子调用函数
借助pybind11的PYBIND11_MODULE宏，将C++实现的npu_add_custom_template函数，封装成可供Python导入和调用的扩展模块接口。m.def三个参数分别是：

- "npu_add_custom_template"：Python 中要调用的函数名（可以和 C++ 函数名不一样，比如写成 "npu_add"，Python 就用 模块名.npu_add() 调用）

- &npu_add_custom_template：C++ 函数的地址 / 指针，告诉 pybind11 要导出的是哪一个 C++ 函数

- "torch add"：函数的文档字符串（可选），这里的torch add是指c++函数的作用类似torch的add。

In [ ]:
%%writefile -a Sources/03.03/pybind_op/custom_op.cpp
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
		m.def("npu_add_custom_template", &npu_add_custom_template, "torch add");
}

#### 4.2.4 编译pybind封装的扩展模块
完成pybind封装代码编写后，我们需要编译成扩展模块，供Python调用。写法比较固定，主要是通过setuptools库的setup函数，指定扩展模块的名称、版本、作者、描述等信息，以及需要编译的源文件。上文pybind绑定的代码我们存放在custom_op.cpp文件中，所以需要在setup.py中指定该文件作为源文件。这里我们指定编译出的模块名为custom_ops_lib， 这样我们可以在Python里通过import custom_ops_lib来使用上文PYBIND11_MODULE宏对外暴露的python接口

In [ ]:
%%writefile Sources/03.03/pybind_op/setup.py
import os
import torch
from setuptools import setup, find_packages
from torch.utils.cpp_extension import BuildExtension

import torch_npu
from torch_npu.utils.cpp_extension import NpuExtension

PYTORCH_NPU_INSTALL_PATH = os.path.dirname(os.path.abspath(torch_npu.__file__))
exts = []
ext1 = NpuExtension(
    name="custom_ops_lib",
    sources=["./custom_op.cpp"],
    extra_compile_args = [
        '-I' + os.path.join(PYTORCH_NPU_INSTALL_PATH, "include/third_party/acl/inc"),
    ],
)
exts.append(ext1)

setup(
    name="custom_ops",
    ext_modules=exts,
    version='1.0',
    cmdclass={"build_ext": BuildExtension},
)

尝试执行以下代码完成pybind封装的扩展模块的编译和安装

In [ ]:
!export LD_LIBRARY_PATH=${HOME}/vendors/customize/op_api/lib/:$LD_LIBRARY_PATH;cd Sources/03.03/pybind_op/;python3 setup.py build bdist_wheel;pip3 install dist/custom_ops*.whl --force-reinstall

### 4.3 python侧调用代码
安装好pybind绑定的扩展模块后，我们可以通过import custom_ops_lib来导入编译出的扩展模块，然后就可以像调用普通Python函数一样调用自定义算子的接口了。需要注意的是自定义算子运行在npu上，所以调用时需要将输入数据转换为npu张量，输出数据需要打印或者保存时也需要先转换为cpu张量。

In [ ]:
%%writefile Sources/03.03/pybind_op/test_op.py

# 我们用Python调用Pybind封装好的npu_add_custom_template函数，以此来运行我们开发的自定义算子
import torch
import torch_npu
import custom_ops_lib

torch.npu.config.allow_internal_format = False

length = [8, 2048]
x = torch.rand(length, device='cpu', dtype=torch.float16)
y = torch.rand(length, device='cpu', dtype=torch.float16)
golden = x + y
output_npu = custom_ops_lib.npu_add_custom_template(x.npu(), y.npu())

print("is same:",torch.allclose(golden, output_npu.cpu(),rtol=0.001, atol=0.001))

In [ ]:
!source ${HOME}/vendors/customize/bin/set_env.bash; python Sources/03.03/pybind_op/test_op.py

python代码运行后，应输出：
```
is same: True
```

---

## **课后实践**
尝试修改下面的单算子API调用代码，将其修改成测试输入为shape [128, 256], 数据类型为float32。

In [ ]:
%%writefile Sources/03.03/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_add_custom_template.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(aclFloat16);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<aclFloat16> input0HostData(elementCount, aclFloatToFloat16(1.0));
    std::vector<aclFloat16> input1HostData(elementCount, aclFloatToFloat16(2.0));
    std::vector<aclFloat16> output0HostData(elementCount, aclFloatToFloat16(0.0));
    std::vector<aclFloat16> goldenData(elementCount, aclFloatToFloat16(3.0));

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnAddCustomTemplateGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnAddCustomTemplate(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", aclFloat16ToFloat(output0HostData[i])); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

修改完成后，执行以下代码测试算子调用是否正确

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/03.03/custom_op

# 复制上一节课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op Sources/03.03/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/03.03/custom_op/CMakePresets.json

# 编译部署算子
!cd Sources/03.03/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

# 部署算子后编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/03.03/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/03.03/execute_add_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/03.03/execute_add_op

**执行以下代码获取答案**

In [ ]:
!cat ./answer/03.03_answer.txt